# MedGemma medical chat assistant — local experiment

This notebook mirrors the flow in `medical_chat_assistant_demo.ipynb` (skin-lesion triage, continuation, optional image review) but runs **`google/medgemma-1.5-4b-it`** locally via the Transformers **`image-text-to-text`** pipeline.

**Setup (gated model):** [MedGemma on Hugging Face](https://huggingface.co/google/medgemma-1.5-4b-it) is **restricted**. You must sign in, **request access**, accept the terms, and **wait until your account is approved** (not instant). Then put a token from **that same account** in `.env` as **`HF_TOKEN`** or **`HUGGING_FACE_HUB_TOKEN`**.

- **401** → missing/invalid token. **403** → either not approved for the model yet, wrong HF account for the token, or (common) a **fine-grained** token without **“Access to public gated repositories”** — fix under [token settings](https://huggingface.co/settings/tokens) or use a classic **Read** token. Run the **“Verify Hugging Face access”** cell below; if Transformers later says *offline / check internet* after a 403, that message is misleading.

Optional: override the repo id with env **`MEDGEMMA_MODEL_ID`**. Do not commit tokens.

**Behavior:** After `reset_conversation()`, the **first** user turn (text or image) includes the full system-style instructions so MedGemma stays aligned with the skin-lesion assistant scope; later turns reuse the pipeline’s returned message list for context.

**Disclaimer:** Demonstration only — not a licensed clinician; not for sole diagnosis or treatment.

## Dependencies

`pip install transformers torch pillow accelerate python-dotenv requests huggingface_hub`

In [1]:
from __future__ import annotations

import json
import os
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Optional

import requests
import torch
from dotenv import load_dotenv
from PIL import Image
from transformers import pipeline

_here = Path.cwd().resolve()
PROJECT_ROOT = _here.parent if _here.name == "notebooks" else _here
load_dotenv(PROJECT_ROOT / ".env")

os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN") or os.getenv("HUGGING_FACE_HUB_TOKEN")


MEDGEMMA_MODEL_ID = os.getenv("MEDGEMMA_MODEL_ID", "google/medgemma-1.5-4b-it")

/home/nipuna/miniconda3/envs/chatllm/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Verify Hugging Face access

Run this **before** the pipeline cell. It prints which account your token belongs to and tries to download `config.json`.

**Fine-grained tokens:** If you see a 403 about *“enable access to public gated repositories”*, open [Hugging Face token settings](https://huggingface.co/settings/tokens), edit the token, and turn on **Access to public gated repositories** (or use a **classic** token with Read access).

If the final error mentions *offline mode* or *check your internet*, that is often a **misleading** follow-on from Transformers when the Hub actually returned **403** — fix token or model access first.

In [3]:
device = 0 if torch.cuda.is_available() else -1
torch_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

pipe = pipeline(
    "image-text-to-text",
    model=MEDGEMMA_MODEL_ID,
    torch_dtype=torch_dtype,
    device=device,
)
print("Model:", MEDGEMMA_MODEL_ID, "| device:", "cuda" if device == 0 else "cpu")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights: 100%|██████████| 883/883 [00:00<00:00, 4959.45it/s]


Model: google/medgemma-1.5-4b-it | device: cuda


In [4]:
MEDICAL_ASSISTANT_INSTRUCTIONS = """You are a supportive health-information assistant focused on **skin cancer / suspicious skin lesions**.

Scope (skin):
- Help users discuss skin spots/moles/rashes/lesions in a structured way.
- Provide **triage-style guidance** (what to monitor, what to ask a clinician, when to seek urgent evaluation).
- Support longitudinal tracking (size, color, symptoms) and healthy sun-safety habits.

Safety and boundaries:
- You are NOT a doctor. Do NOT provide a definitive diagnosis (e.g., do not say "this is melanoma").
- Make it clear that skin cancer assessment often requires an in-person exam and sometimes dermoscopy/biopsy.
- Use cautious language ("could be", "possible", "worth getting checked").
- If there are urgent red flags (rapidly changing lesion, bleeding/ulceration, severe pain, signs of infection, systemic symptoms, immunosuppression + concerning lesion), advise prompt in-person care.
- If the user reports severe symptoms or is otherwise unwell, advise urgent/emergency care as appropriate.

Image handling (when an image is provided):
- Describe only what is visible. Do not claim certainty from a photo.
- Use evidence-based framing like **ABCDE** (Asymmetry, Border, Color, Diameter, Evolving) and the "ugly duckling" sign.
- Ask for key context: duration, changes, symptoms (itching, pain, bleeding), location, personal/family history, sun exposure, and any prior skin cancers.

Style:
- Be empathetic, concise, and clear. Ask clarifying questions when it improves safety and usefulness.
- When recommending next steps, be specific (e.g., "book a dermatology visit", "take standardized photos weekly", "measure with a ruler").
- Keep replies short: about 50–75 words per message when possible.
"""


@dataclass
class PatientProfile:
    display_name: str = "Alex"
    age: Optional[int] = None
    chronic_conditions: str = ""
    monitoring_goals: str = ""
    communication_style: str = "warm, encouraging mentor"
    extra_notes: str = ""

    def personalization_block(self) -> str:
        parts = [
            f"Address the user as {self.display_name}.",
            f"Communication style: {self.communication_style}.",
        ]
        if self.age is not None:
            parts.append(f"Patient-stated age: {self.age}.")
        if self.chronic_conditions.strip():
            parts.append(f"Patient-stated conditions / history: {self.chronic_conditions.strip()}")
        if self.monitoring_goals.strip():
            parts.append(f"Current monitoring / goals: {self.monitoring_goals.strip()}")
        if self.extra_notes.strip():
            parts.append(f"Additional context: {self.extra_notes.strip()}")
        return "\n".join(parts)


def build_instructions(profile: PatientProfile) -> str:
    return (
        MEDICAL_ASSISTANT_INSTRUCTIONS
        + "\n\n---\nPersonalization:\n"
        + profile.personalization_block()
    )

In [5]:
def _content_to_str(content: Any) -> str:
    if content is None:
        return ""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        texts = []
        for block in content:
            if isinstance(block, dict) and block.get("type") == "text":
                texts.append(str(block.get("text", "")))
        return "\n".join(texts) if texts else ""
    return str(content)


def _extract_assistant_reply(generated: list[dict[str, Any]]) -> str:
    if not generated:
        return ""
    last = generated[-1]
    role = last.get("role")
    if role in ("assistant", "model"):
        return _content_to_str(last.get("content")).strip()
    return _content_to_str(last.get("content")).strip()


class MedGemmaMedicalChatSession:
    """Multi-turn skin-lesion assistant using local MedGemma (Transformers pipeline)."""

    def __init__(
        self,
        medgemma_pipe: Any,
        profile: PatientProfile,
        *,
        max_new_tokens: int = 512,
    ) -> None:
        self.pipe = medgemma_pipe
        self.profile = profile
        self.max_new_tokens = max_new_tokens
        self.transcript: list[dict[str, Any]] = []
        self.messages: list[dict[str, Any]] = []
        self.reset_conversation()

    @property
    def instructions(self) -> str:
        return build_instructions(self.profile)

    def reset_conversation(self) -> None:
        # Empty thread: first user turn prepends full instructions (MedGemma chat templates
        # often omit a separate system role; prior turns stay in self.messages for context).
        self.messages = []

    def _is_new_thread(self) -> bool:
        return len(self.messages) == 0

    def _prefix_first_turn(self, user_text: str) -> str:
        return f"{self.instructions}\n\n---\n\n{user_text}"

    def _append_transcript(self, role: str, content: str, meta: Optional[dict[str, Any]] = None) -> None:
        entry: dict[str, Any] = {
            "role": role,
            "content": content,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
        if meta:
            entry["meta"] = meta
        self.transcript.append(entry)

    def _normalize_content(self, content: Any) -> list[dict[str, Any]]:
        if isinstance(content, str):
            return [{"type": "text", "text": content}]
        if isinstance(content, dict):
            return [content]
        if isinstance(content, list):
            normalized: list[dict[str, Any]] = []
            for block in content:
                if isinstance(block, str):
                    normalized.append({"type": "text", "text": block})
                elif isinstance(block, dict):
                    if block.get("type") == "text" and "text" not in block:
                        block = {**block, "text": ""}
                    normalized.append(block)
                else:
                    normalized.append({"type": "text", "text": str(block)})
            return normalized
        return [{"type": "text", "text": str(content)}]

    def _normalize_messages(self) -> list[dict[str, Any]]:
        normalized: list[dict[str, Any]] = []
        for msg in self.messages:
            role = msg.get("role", "user") if isinstance(msg, dict) else "user"
            content = msg.get("content", "") if isinstance(msg, dict) else str(msg)
            normalized.append({"role": role, "content": self._normalize_content(content)})
        return normalized

    def _generate(self) -> str:
        self.messages = self._normalize_messages()
        out = self.pipe(text=self.messages, max_new_tokens=self.max_new_tokens)
        generated = out[0]["generated_text"]
        self.messages = generated
        return _extract_assistant_reply(generated)

    def chat(self, user_text: str) -> str:
        payload = self._prefix_first_turn(user_text) if self._is_new_thread() else user_text
        self.messages.append({"role": "user", "content": [{"type": "text", "text": payload}]})
        reply = self._generate()
        self._append_transcript("user", user_text)
        self._append_transcript("assistant", reply)
        return reply

    def chat_with_image(
        self,
        user_text: str,
        *,
        image_path: Optional[str | Path] = None,
        image_url: Optional[str] = None,
        pil_image: Optional[Image.Image] = None,
    ) -> str:
        if pil_image is not None:
            image = pil_image
        elif image_path is not None:
            image = Image.open(Path(image_path)).convert("RGB")
        elif image_url is not None:
            image = Image.open(
                requests.get(image_url, headers={"User-Agent": "medgemma-notebook"}, stream=True).raw
            ).convert("RGB")
        else:
            raise ValueError("Provide pil_image, image_path, or image_url")

        payload = self._prefix_first_turn(user_text) if self._is_new_thread() else user_text
        self.messages.append(
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": payload},
                ],
            }
        )
        reply = self._generate()
        self._append_transcript("user", user_text, meta={"has_image": True})
        self._append_transcript("assistant", reply)
        return reply

    def save_transcript(self, path: str | Path) -> Path:
        path = Path(path)
        path.write_text(
            json.dumps(
                {
                    "backend": "medgemma",
                    "model": MEDGEMMA_MODEL_ID,
                    "profile": self.profile.__dict__,
                    "transcript": self.transcript,
                },
                indent=2,
                ensure_ascii=False,
            ),
            encoding="utf-8",
        )
        return path

In [6]:
profile = PatientProfile(
    display_name="Alex",
    age=42,
    chronic_conditions="Seasonal allergies (patient-reported).",
    monitoring_goals="Daily energy and sleep check-in for two weeks.",
    communication_style="warm mentor; short paragraphs",
)

session = MedGemmaMedicalChatSession(pipe, profile)
print("Instructions preview:\n", session.instructions[:500], "...")

Instructions preview:
 You are a supportive health-information assistant focused on **skin cancer / suspicious skin lesions**.

Scope (skin):
- Help users discuss skin spots/moles/rashes/lesions in a structured way.
- Provide **triage-style guidance** (what to monitor, what to ask a clinician, when to seek urgent evaluation).
- Support longitudinal tracking (size, color, symptoms) and healthy sun-safety habits.

Safety and boundaries:
- You are NOT a doctor. Do NOT provide a definitive diagnosis (e.g., do not say "thi ...


## Demo: skin lesion check-in (same prompt family as `medical_chat_assistant_demo.ipynb`)

In [7]:
import time

t0 = time.time()
reply = session.chat(
    "Skin check-in: I noticed a mole on my upper arm that seems a bit darker than the others. "
    "I'm not sure if it has changed, but it's about the size of a pencil eraser. It doesn't hurt. "
    "What questions should you ask me to assess risk, and what should I monitor over the next 2 weeks?"
)
t1 = time.time()
print(reply)
print(f"Inference time: {t1 - t0:.2f}s")

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Okay, Alex, thanks for sharing that. It's good you're paying attention to changes in your skin. Let's break this down.

To help me understand better, could you tell me:

1.  **When did you first notice this mole?**
2.  **Has it changed in size, shape, color, or texture?**
3.  **Does it itch, hurt, bleed, or crust?**
4.  **How much sun exposure do you typically get on your upper arm?**
5.  **Have you ever had skin cancer before?**

Over the next two weeks, I recommend:

1.  **Daily observation:** Look at it closely each day. Note any changes in size, shape, color, or if it starts to itch, bleed, or change texture.
2.  **Standardized photos:** Take clear, well-lit photos of the mole from different angles (front, back, side) using your phone.
3.  **Measurement:** Use a ruler to measure its diameter (width and length) in millimeters.

If you notice any significant changes, or if the mole starts to bleed or itch, please schedule an appointment with a dermatologist or your primary care physi

## Continuation (multi-turn context in `session.messages`)

In [8]:
follow_up = session.chat(
    "Update: I compared it to my other moles and this one looks a bit more irregular at the edges. "
    "It also seems like it might have gotten slightly darker over the last month (not sure). "
    "What specific red flags should push me to book a dermatology visit sooner rather than later?"
)
print(follow_up)

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<unused94>thought
The user is reporting two new observations about the mole:
1.  **Irregular edges:** This is a potential sign of change, which is a key component of the ABCDEs of melanoma.
2.  **Possible darkening:** Another potential sign of change, also relevant to ABCDEs.

The user is asking for specific red flags that would warrant a sooner dermatology visit.

My goal is to provide triage-style guidance based on the user's report, focusing on urgency and the need for professional evaluation, without diagnosing. I need to use cautious language and emphasize the importance of seeing a doctor.

Key points to cover:
-   Acknowledge the user's observations (irregular edges, possible darkening).
-   Explain *why* these are potentially concerning (changes are key).
-   List specific red flags that would necessitate prompt evaluation (rapid change, bleeding, itching, ulceration, significant size increase, etc.).
-   Reiterate the recommendation to see a dermatologist or PCP.
-   Maintain 

## Optional: local skin photo (replace path; de-identify real patient images)

Uses the same `org_2_5.jpg` convention as the OpenAI demo when present.

In [10]:
DEMO_IMAGE_PATH = PROJECT_ROOT / "org_2_5.jpg"
if DEMO_IMAGE_PATH.is_file():
    session.reset_conversation()
    vision_reply = session.chat_with_image(
        "Please look at this skin photo and do a careful, non-diagnostic review using the ABCDE framework. "
        "List what would make this more vs less concerning, what extra context you need, and what the safest next steps are. "
        "Do NOT provide a definitive diagnosis from the image.",
        image_path=DEMO_IMAGE_PATH,
    )
    print(vision_reply)
else:
    print("Skip: not found —", DEMO_IMAGE_PATH)

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Hi Alex, thanks for sharing this image. Let's take a look together.

**Reviewing the image:**

*   **Asymmetry:** The lesion appears somewhat asymmetrical.
*   **Border:** The border seems slightly irregular.
*   **Color:** The color is mostly brown, but there might be some slight variations.
*   **Diameter:** It's hard to tell the exact size without a scale, but it's a noticeable spot.
*   **Evolving:** I can't assess evolution from a single image.

**What makes it potentially more concerning:**

*   The slight asymmetry and irregular border are features that warrant attention.

**What makes it potentially less concerning:**

*   The color is relatively uniform.

**Context needed:**

*   **Duration:** How long has this spot been present?
*   **Changes:** Has it changed in size, shape, color, or texture?
*   **Symptoms:** Does it itch, bleed, or hurt?
*   **Location:** Where on your body is this located?
*   **History:** Do you have any personal or family history of skin cancer?
*   **

## Export transcript to JSON

In [11]:
out_path = PROJECT_ROOT / "notebooks" / "medical_chat_transcript_medgemma.json"
saved = session.save_transcript(out_path)
print("Saved:", saved)

Saved: /home/nipuna/medical_chat_bot_dev/notebooks/medical_chat_transcript_medgemma.json


## New thread (`reset_conversation`)

In [12]:
session.reset_conversation()
print(session.chat("Let's start a new topic: how do I prepare questions for my annual physical?"))

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Okay, Alex, that's a great idea! Preparing questions for your annual physical can help you get the most out of your visit. Think of it like preparing for a conversation with a knowledgeable friend.

Here are some areas to consider, along with some example questions you could ask your doctor:

**1. General Health & Screening:**
*   "What screenings are recommended for my age and health profile?" (This covers things like mammograms, colonoscopies, Pap smears, etc.)
*   "Are there any new guidelines or recommendations I should be aware of?"

**2. Specific Concerns (like your skin checks):**
*   "How often should I be checking my skin for changes?"
*   "What specific signs should I look for that warrant a doctor's evaluation?"
*   "Do you have any recommendations for skin cancer prevention or early detection?"

**3. Lifestyle & Habits:**
*   "Based on my lifestyle, what are the key health risks I should be mindful of?"
*   "Are there any lifestyle changes you'd recommend for my overall wel